# V2a-RSNs Clustered IC Selection BSS Analysis

This notebook duplicates the shared linear BSS workflow for FastICA, Infomax, SOBI, and JADE,
but selects rejected ICs from spectral clusters instead of manually listing component indices.
Choose `DATASET_KEY`, `METHODS_TO_RUN`, and the cluster-selection settings below. Outputs are saved
under `outputs/clustered/<dataset_group>/<data_name>/<method>/` when `SAVE_OUTPUTS = True`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    p
    for p in candidates
    if (p / "pyproject.toml").exists() and (p / "src" / "bss_notebook.py").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
CORE_DIR = SRC_DIR / "core"
for path in (SRC_DIR, CORE_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from bss_notebook import (
    BSS_METHODS,
    available_datasets,
    load_traces,
    output_directory,
    run_many_methods,
    save_bss_outputs,
    summarize_results,
)
from ica_utils import reconstruct_bss
from visualization import plot_clusters

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PROJECT_ROOT


In [ ]:
available_datasets(PROJECT_ROOT)


In [ ]:
# Dataset and method selection
DATASET_KEY = "v2a-RSNs/220127_F4_run2_fluorescence"

# Use list(BSS_METHODS) to run all four methods, or choose a subset such as ["fastica"].
METHODS_TO_RUN = list(BSS_METHODS)

# None uses the dataset default from bss_notebook.dataset_registry().
N_COMPONENTS = None
DECOMPOSITION_TOL = 0.0001
DECOMPOSITION_MAX_ITER = 500
DECOMPOSITION_RANDOM_STATE = 0

# Cluster ICs by Welch-spectrum features, matching the reference ICA notebook's clustering workflow.
N_CLUSTERS = 7
CLUSTER_FEATURE_START_BIN = 30
CLUSTER_NPERSEG = 250
CLUSTER_NOVERLAP = 125

# Fill after inspecting the 3D cluster plots. If keep_clusters is non-empty, all other clusters are rejected.
CLUSTER_SELECTION_BY_METHOD = {
    method: {"reject_clusters": [], "keep_clusters": []}
    for method in BSS_METHODS
}

# Keep False until cluster rejection choices are final.
SAVE_OUTPUTS = False


In [ ]:
dataset, traces = load_traces(DATASET_KEY, PROJECT_ROOT)
n_components = min(N_COMPONENTS or dataset.default_n_components or min(traces.shape), *traces.shape)
n_neurons, n_frames = traces.shape
sample_rate_hz = dataset.sample_rate_hz or 1.0

print(f"dataset: {dataset.key}")
print(f"trace file: {dataset.trace_path.relative_to(PROJECT_ROOT)}")
print(f"traces: neurons={n_neurons}, frames={n_frames}")
print(f"methods: {METHODS_TO_RUN}")
print(f"n_components: {n_components}")
print(f"sample_rate_hz: {sample_rate_hz}")
for method in METHODS_TO_RUN:
    out_dir = output_directory(
        method,
        dataset.data_name,
        PROJECT_ROOT,
        analysis_kind="clustered",
        dataset_group=dataset.group,
    )
    print(f"{method} output dir: {out_dir.relative_to(PROJECT_ROOT)}")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
time_slice = slice(0, min(300, n_frames))
offset = np.nanstd(traces[:, time_slice]) * .4
if not np.isfinite(offset) or offset == 0:
    offset = 1.0
for neuron_idx in range(min(6, n_neurons)):
    ax.plot(
        np.arange(time_slice.start, time_slice.stop),
        traces[neuron_idx, time_slice] + neuron_idx * offset,
        lw=0.8,
        label=f"neuron {neuron_idx}",
    )
ax.set_title("Raw traces")
ax.set_xlabel("frame")
ax.legend(loc="upper right", ncols=3);


In [ ]:
results = run_many_methods(
    dataset_key=DATASET_KEY,
    methods=METHODS_TO_RUN,
    n_components=N_COMPONENTS,
    reject_components_by_method={},
    save_outputs=False,
    project_root=PROJECT_ROOT,
    tol=DECOMPOSITION_TOL,
    max_iter=DECOMPOSITION_MAX_ITER,
    random_state=DECOMPOSITION_RANDOM_STATE,
    analysis_kind="clustered",
)

summarize_results(results)


In [ ]:
component_count = min(8, n_components)
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.0 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]
for ax, (method, result) in zip(axes, results.items()):
    for comp_idx in range(component_count):
        ax.plot(result.ic_comps[:, comp_idx] + comp_idx * 3, lw=0.7, label=f"IC {comp_idx}")
    ax.set_title(f"{method.upper()} first {component_count} components")
axes[-1].set_xlabel("frame");


In [ ]:
def cluster(
    ICs,
    n_clusters,
    sample_rate_hz,
    *,
    feature_start_bin=30,
    nperseg=250,
    noverlap=125,
    random_state=0,
):
    """Cluster ICs by truncated Welch spectra and project them to 3D for plotting.

    Parameters
    ----------
    ICs : array, shape (frames, n_components)
        Time-domain independent components returned by the BSS decomposition.
    n_clusters : int
        Number of KMeans clusters to request. Values above the component count are clipped.
    sample_rate_hz : float
        Recording sample rate used for Welch spectra.
    feature_start_bin : int
        Lower spectral bin removed before KMeans/PCA, matching the reference notebook's `[:, 30:]`.
    """
    ICs = np.asarray(ICs, dtype=float)
    if ICs.ndim != 2:
        raise ValueError(f"ICs must have shape (frames, components), got {ICs.shape}.")

    n_frames, n_ics = ICs.shape
    if n_ics < 1:
        raise ValueError("ICs must contain at least one component.")

    nperseg = min(int(nperseg), n_frames)
    noverlap = min(int(noverlap), max(nperseg - 1, 0))
    spectra = []
    for ic_idx in range(n_ics):
        _, power = welch(
            ICs[:, ic_idx],
            fs=sample_rate_hz,
            return_onesided=True,
            nperseg=nperseg,
            noverlap=noverlap,
        )
        spectra.append(power)
    spectra = np.asarray(spectra)

    start_bin = min(max(int(feature_start_bin), 0), max(spectra.shape[1] - 1, 0))
    features = np.log1p(spectra[:, start_bin:])
    if features.shape[1] < 3:
        features = np.log1p(spectra)

    n_clusters = min(int(n_clusters), n_ics)
    if n_clusters < 1:
        raise ValueError("n_clusters must be at least 1.")

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    predictions = kmeans.fit_predict(features)

    n_pca_components = min(3, features.shape[0], features.shape[1])
    new_mat = PCA(n_components=n_pca_components).fit_transform(features)
    if n_pca_components < 3:
        new_mat = np.pad(new_mat, ((0, 0), (0, 3 - n_pca_components)), constant_values=0.0)

    return new_mat, predictions, spectra, features


In [ ]:
cluster_results = {}
cluster_rows = []

for method, result in results.items():
    new_mat, predictions, spectra, features = cluster(
        result.ic_comps,
        N_CLUSTERS,
        sample_rate_hz,
        feature_start_bin=CLUSTER_FEATURE_START_BIN,
        nperseg=CLUSTER_NPERSEG,
        noverlap=CLUSTER_NOVERLAP,
        random_state=DECOMPOSITION_RANDOM_STATE,
    )
    cluster_results[method] = {
        "new_mat": new_mat,
        "predictions": predictions,
        "spectra": spectra,
        "features": features,
    }
    for cluster_id in sorted(np.unique(predictions)):
        ic_indices = np.flatnonzero(predictions == cluster_id)
        cluster_rows.append(
            {
                "method": method,
                "cluster": int(cluster_id),
                "n_ics": int(ic_indices.size),
                "ic_indices": ic_indices.tolist(),
            }
        )

pd.DataFrame(cluster_rows)


In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "notebook")
except Exception:
    pass

for method, clustered in cluster_results.items():
    print(method)
    plot_clusters(clustered["new_mat"], clustered["predictions"])


In [ ]:
def reject_components_from_cluster_selection(predictions, *, reject_clusters=(), keep_clusters=()):
    """Convert cluster label selections into component indices to reject."""
    predictions = np.asarray(predictions)
    reject_clusters = list(reject_clusters)
    keep_clusters = list(keep_clusters)
    if reject_clusters and keep_clusters:
        raise ValueError("Use reject_clusters or keep_clusters for a method, not both.")
    if keep_clusters:
        keep_mask = np.isin(predictions, keep_clusters)
        return np.flatnonzero(~keep_mask)
    reject_mask = np.isin(predictions, reject_clusters)
    return np.flatnonzero(reject_mask)

clustered_cleaned = {}
reject_components_by_method = {}
selection_rows = []

for method, result in results.items():
    selection = CLUSTER_SELECTION_BY_METHOD.get(method, {})
    predictions = cluster_results[method]["predictions"]
    reject_components = reject_components_from_cluster_selection(
        predictions,
        reject_clusters=selection.get("reject_clusters", []),
        keep_clusters=selection.get("keep_clusters", []),
    )
    reject_components_by_method[method] = reject_components.tolist()
    clustered_cleaned[method] = reconstruct_bss(
        result.ic_comps,
        result.A,
        result.mean,
        reject=reject_components,
    )
    selection_rows.append(
        {
            "method": method,
            "reject_clusters": selection.get("reject_clusters", []),
            "keep_clusters": selection.get("keep_clusters", []),
            "reject_components": reject_components.tolist(),
            "n_rejected": int(reject_components.size),
        }
    )

pd.DataFrame(selection_rows)


In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.3 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]
neuron_idx = 0
for ax, (method, result) in zip(axes, results.items()):
    ax.plot(traces[neuron_idx, time_slice], lw=0.8, label="raw", color="red", alpha=0.65)
    ax.plot(
        clustered_cleaned[method][time_slice, neuron_idx],
        lw=0.8,
        label=f"{method} cluster-cleaned",
        color="blue",
        alpha=0.8,
    )
    ax.set_title(f"Raw vs cluster-cleaned, neuron {neuron_idx}, {method}")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("frame");


In [ ]:
saved_paths_by_method = {}

if SAVE_OUTPUTS:
    for method, result in results.items():
        out_dir = output_directory(
            method,
            result.dataset.data_name,
            PROJECT_ROOT,
            analysis_kind="clustered",
            dataset_group=result.dataset.group,
        )
        saved_paths_by_method[method] = save_bss_outputs(
            spec=result.dataset,
            method=method,
            traces=result.traces,
            ic_comps=result.ic_comps,
            IC_ft=result.IC_ft,
            A=result.A,
            mean=result.mean,
            cleaned=clustered_cleaned[method],
            reject_components=reject_components_by_method[method],
            output_dir=out_dir,
            n_components=result.ic_comps.shape[1],
            tol=DECOMPOSITION_TOL,
            max_iter=DECOMPOSITION_MAX_ITER,
            random_state=DECOMPOSITION_RANDOM_STATE,
        )

for method, result in results.items():
    if method in saved_paths_by_method:
        print(f"{method}:")
        for label, path in saved_paths_by_method[method].items():
            print(f"  {label}: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(
            f"{method}: not saved; set SAVE_OUTPUTS = True after cluster selections are final "
            f"to write outputs under {result.output_dir.relative_to(PROJECT_ROOT)}"
        )
